In [ ]:
# ========================================================
# BÀI 2 (TRANG 15-16) - CÂU 10
# Đề bài: So sánh kết quả của các mô hình trên (ID3, CART, Naïve Bayes).
# ========================================================

# 1. BẮT BUỘC: IN HỌ TÊN VÀ MSSV
HO_TEN = "Đào Tiến Đạt"
MSSV = "6451071017"
print(f"HỌ TÊN: {HO_TEN} - MSSV: {MSSV}")
print("=" * 60)

%matplotlib inline
import matplotlib.pyplot as plt
from sklearn import tree, metrics
from sklearn.naive_bayes import GaussianNB
import os
import pandas as pd
import numpy as np

def find_file(name):
    paths = [
        name,
        os.path.join(r"D:\khai pha du lieu\Lab2\Dataset-20261103T094921Z-1-001\Dataset", name),
        os.path.join("..", "..", "..", "Dataset-20261103T094921Z-1-001", "Dataset", name),
        os.path.join("Dataset", name)
    ]
    for p in paths:
        if os.path.exists(p):
            return p
    return name


# Chuẩn bị dữ liệu
train_data = pd.read_csv(find_file("adult.data.csv"), skipinitialspace=True, na_values='?').dropna()
test_data = pd.read_csv(find_file("adult.test.csv"), skipinitialspace=True, na_values='?').dropna()
num_train = len(train_data)

if 'fnlwgt' in train_data.columns:
    train_data = train_data.drop(columns=['fnlwgt'])
data = pd.concat([train_data, test_data], ignore_index=True)

features = data.drop('income', axis=1)
labels = data['income']
cat_cols = features.select_dtypes(exclude=['int64', 'float64']).columns
features_onehot = pd.get_dummies(features, columns=cat_cols)

X_train = features_onehot[:num_train]
X_test = features_onehot[num_train:]
y_train = labels[:num_train]
y_test = labels[num_train:]

# 1. Cây ID3
clf_id3 = tree.DecisionTreeClassifier(criterion='entropy', random_state=42)
clf_id3.fit(X_train, y_train)
pred_id3 = clf_id3.predict(X_test)
acc_id3 = metrics.accuracy_score(y_test, pred_id3)
f1_id3 = metrics.f1_score(y_test, pred_id3, average='weighted')

# 2. Cây CART
clf_cart = tree.DecisionTreeClassifier(criterion='gini', random_state=42)
clf_cart.fit(X_train, y_train)
pred_cart = clf_cart.predict(X_test)
acc_cart = metrics.accuracy_score(y_test, pred_cart)
f1_cart = metrics.f1_score(y_test, pred_cart, average='weighted')

# 3. Naïve Bayes
clf_gnb = GaussianNB()
clf_gnb.fit(X_train, y_train)
pred_gnb = clf_gnb.predict(X_test)
acc_gnb = metrics.accuracy_score(y_test, pred_gnb)
f1_gnb = metrics.f1_score(y_test, pred_gnb, average='weighted')

# 4. Bảng tổng hợp so sánh
df_compare = pd.DataFrame({
    'Model': ['Decision Tree (ID3)', 'Decision Tree (CART)', 'Gaussian Naïve Bayes'],
    'Accuracy': [acc_id3, acc_cart, acc_gnb],
    'F1-Score (Weighted)': [f1_id3, f1_cart, f1_gnb]
})
print("BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH:")
display(df_compare)

# 5. Trực quan hóa so sánh
plt.figure(figsize=(10, 5))
bars = plt.bar(df_compare['Model'], df_compare['Accuracy'], color=['royalblue', 'mediumseagreen', 'coral'], width=0.45)
plt.ylim(0.7, 0.9)
plt.ylabel('Độ chính xác (Accuracy)')
plt.title('So sánh độ chính xác giữa ID3, CART và Naïve Bayes trên tập Adult Income')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 6. Nhận xét chi tiết
print("=" * 60)
print("NHẬN XÉT VÀ KẾT LUẬN:")
print("1. Cây quyết định CART và ID3 đạt độ chính xác tương đương nhau (~81.5% - 81.8%), cao hơn mô hình Naïve Bayes (~80.3%).")
print("2. Cây quyết định học tốt các quy luật phi tuyến tính và sự tương tác giữa các thuộc tính phân loại (categorical attributes).")
print("3. Naïve Bayes giả định độc lập có điều kiện, nhưng trong thực tế các thuộc tính nhân khẩu học (tuổi, trình độ học vấn, giờ làm việc) có mối liên hệ mật thiết, làm giảm độ chính xác một phần.")
